# Get Raw Data

- Reads the notes directly from BigQuery where is the MIMIC-IV database and the discharge dataset that has the discharge notes; 
- Pulls out the two admission and discharge medication sections with regex, and writes the result to a specific folder.

**Note:** 
- this step does NOT use an LLM. 
- The extraction is plain text-matching and runs entirely on this machine;

## 1. Setup & Imports

In [ ]:
import os
import pandas as pd

from pathlib import Path

from clinical_notes_extraction.config import PROJECT_ROOT, QUERY_FILE_NAME
from clinical_notes_extraction.utils.io import fetch_data
from clinical_notes_extraction.utils.extraction import add_medication_columns



In [ ]:
final_path = PROJECT_ROOT/'data'
# Creates the entire folder structure; does nothing if they already exist
os.makedirs(final_path, exist_ok=True)

raw_output_path = f'{final_path}/raw'
os.makedirs(raw_output_path, exist_ok=True)


datasets_output_path = f'{final_path}/datasets'
os.makedirs(datasets_output_path, exist_ok=True)

## 2. Extract Discharge Notes from MIMIC-IV

In [ ]:
raw_dataset = Path(f'{raw_output_path}/discharge_notes_with_meds.parquet')

# Query BigQuery only once: on later runs the local parquet is read instead.
if raw_dataset.exists():
    df = pd.read_parquet(raw_dataset)
    print(f"Loaded {len(df):,} rows from cache")
else:
    df = fetch_data(QUERY_FILE_NAME)
    print(f"Fetched {len(df):,} rows and cached them")

In [ ]:
df.head()

In [ ]:
df.info()

## 3. Extract Admission and Discharge Medication Sections from Discharge Notes

- Pulls out the two admission and discharge medication sections with regex, and writes the result to DATA_PATH directory.
- Medication Sections are described in discharge notes text as:
    - "Medications on Admission"
    - "Discharge Medications"

In [ ]:
df1 = add_medication_columns(df)

In [ ]:
df1.head()

In [ ]:
df1.info()

## 4. Save the dataset containg all raw data of clinical notes

**Save raw dataset:**

In [ ]:
raw_dataset = f'{raw_output_path}/discharge_notes_with_meds.parquet'

df.to_parquet(raw_dataset, index=False)

print(f"Wrote {len(df)} rows to {raw_dataset}" )

**Save medication_on_admission dataset:**

In [ ]:
df1.info()

In [ ]:
df_meds_on_admission = df1.drop(columns='meds_on_discharge')

df_meds_on_admission.info()

In [ ]:
meds_admission_dataset_path = f'{datasets_output_path}/medication_on_admission.parquet'

df_meds_on_admission.to_parquet(meds_admission_dataset_path, index=False)  

print(f"Wrote {len(df_meds_on_admission)} rows to {meds_admission_dataset_path}" )

**Save medication_on_discharge dataset:**

In [ ]:
df_meds_on_discharge = df1.drop(columns='meds_on_admission')

df_meds_on_discharge.info()

In [ ]:
meds_discharge_dataset_path = f'{datasets_output_path}/medication_on_discharge.parquet'

df_meds_on_discharge.to_parquet(meds_discharge_dataset_path, index=False)  

print(f"Wrote {len(df_meds_on_discharge)} rows to {meds_discharge_dataset_path}" )